In [1]:
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('../data/processed/feature_engineered_shots.csv')
df.head()

,minute,second,period,team_name,player_name,position_name,x,y,end_x,end_y,...,shot_one_on_one,shot_open_goal,shot_deflected,shot_statsbomb_xg,outcome_name,season_name,goal,shot_distance,angle,centrality
0,0,18,1,Switzerland,Granit Xhaka,Left Defensive Midfield,96.0,38.8,108.2,38.5,...,False,False,False,0.036566,Blocked,2022,0,24.029981,0.301942,1.2
1,0,22,1,Switzerland,Breel-Donald Embolo,Center Forward,113.1,40.7,114.8,40.6,...,False,False,False,0.353289,Saved,2022,0,6.935416,0.968776,0.7
2,0,23,1,Switzerland,Granit Xhaka,Left Defensive Midfield,103.8,41.9,115.5,39.1,...,False,False,False,0.069527,Saved,2022,0,16.311039,0.438830,1.9
3,4,35,1,Serbia,Nikola Milenković,Right Center Back,112.2,36.8,120.0,35.3,...,False,False,False,0.081609,Off T,2022,0,8.430896,0.780272,3.2
4,10,5,1,Serbia,Andrija Živković,Right Wing Back,97.8,51.5,120.0,36.1,...,False,False,False,0.030002,Post,2022,0,25.001800,0.259664,11.5


In [3]:
features = [
    'minute',
    'period',
    'x',
    'y',
    'shot_distance',
    'angle',
    'centrality',
    'body_part_name',
    'technique_name',
    'play_pattern_name',
    'under_pressure',
    'shot_first_time',
    'shot_one_on_one',
    'shot_open_goal',
    'shot_deflected'
]

X = df[features]

y = df['goal']

In [4]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify =y
)

In [5]:
numerical_features = [
    'minute',
    'period',
    'x',
    'y',
    'shot_distance',
    'angle',
    'centrality'
]

categorical_features = [
    'body_part_name',
    'technique_name',
    'play_pattern_name',
    'under_pressure',
    'shot_first_time',
    'shot_one_on_one',
    'shot_open_goal',
    'shot_deflected'
]

In [6]:
numerical_transformer = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('impuuter', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown = 'ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer , categorical_features)
    ]
)

In [7]:
# logistic regression cv
logreg_pipeline = Pipeline(
    steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
    ]
)

cv_scores = cross_val_score(
    logreg_pipeline,
    X,
    y,
    cv=5,
    scoring = 'roc_auc'
)

print('Cross Validation ROC-AUC Scores:')
print(cv_scores)
print('\nMean ROC-AUC:')
print(cv_scores.mean())

Cross Validation ROC-AUC Scores:
[0.82470796 0.84861357 0.82438643 0.78403789 0.83922406]

Mean ROC-AUC:
0.8241939820366205


In [8]:
xgb_pipeline = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model', XGBClassifier (
            random_state = 42,
            eval_metric = 'logloss'
        ))
    ]
)

In [9]:
param_grid = {
    'model__n_estimators':[100,200],
    'model__max_depth':[3,4,5],
    'model__learning_rate':[0.01,0.05,0.1]
}

In [10]:
grid_search = GridSearchCV(
    xgb_pipeline,
    param_grid,
    cv=3,
    scoring='roc_auc',
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest ROC-AUC:")
print(grid_search.best_score_)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best Parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100}

Best ROC-AUC:
0.8159896184053431


In [11]:
best_model = grid_search.best_estimator_

y_prob_best = best_model.predict_proba(X_test)[:,1]

final_auc = roc_auc_score(
    y_test,
    y_prob_best
)

print("Final Test ROC-AUC:")
print(final_auc)

Final Test ROC-AUC:
0.8084756438969765


In [12]:
import os

print(os.getcwd())
print(os.listdir("../models"))


C:\Users\joshi\xg-predictor\notebooks
['.ipynb_checkpoints', 'xg_logistic_model.pkl']


In [15]:
logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

logreg_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['minute', 'period', 'x', 'y',
                                                   'shot_distance', 'angle',
                                                   'centrality']),
                                                 ('cat',
                                                  Pipeline(steps=[('impuuter',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['body_part_name',
                                                   'technique_name',
                                                   'play_pattern_name',
                                                   'under_pressure',
                                                   'shot_first_time',
                                                   'shot_one_on_one',
                                                   'shot_open_goal',
                                                   'shot_deflected'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [16]:
sample_predictions = logreg_pipeline.predict_proba(
    X_test.head()
)[:, 1]

print(sample_predictions)

[0.03291242 0.22718121 0.05705541 0.02374432 0.60327902]


In [17]:
import joblib

joblib.dump(
    logreg_pipeline,
    "../models/xg_logistic_model.pkl"
)

['../models/xg_logistic_model.pkl']

In [18]:
loaded_model = joblib.load("../models/xg_logistic_model.pkl")

sample_predictions = loaded_model.predict_proba(
    X_test.head()
)[:, 1]

print(sample_predictions)

[0.03291242 0.22718121 0.05705541 0.02374432 0.60327902]


In [19]:
print(type(loaded_model))


<class 'sklearn.pipeline.Pipeline'>
